# Deep Learning Lab: 1D-CNN Issue Classifier

In this lab, we build, train, and inspect a 1D Convolutional Neural Network (CNN)
for categorizing developer issues from GitHub Archive.

### Architecture:
1. Tokenizer & Embedding Layer (vocab_size -> 64)
2. 1D Convolution (filter size 3, 64 channels)
3. ReLU Non-Linearity
4. Global Max-over-Time Pooling
5. Dropout (0.2) & Linear Classification Head (64 -> 4 classes)

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('.').resolve()))

import pyarrow.parquet as pq
from src.ml.dataset import IssueVocabulary
from src.ml.model import create_issue_cnn

print('DL modules imported successfully!')

## 1. Load Data and Inspect Vocabulary
We build the vocabulary strictly from the training split to prevent data leakage.

In [ ]:
splits_table = pq.read_table('data/curated/splits.parquet')
records = splits_table.to_pylist()
train_texts = [f"{r['title']} {r['body']}" for r in records if r['split'] == 'train']

vocab = IssueVocabulary(min_freq=2)
vocab.build_from_texts(train_texts)
print(f'Training Vocabulary size: {len(vocab):,} unique tokens')

sample = 'Fatal panic in go runtime during garbage collection'
encoded = vocab.encode(sample, max_len=15)
print(f'Sample: {sample}')
print(f'Encoded IDs: {encoded}')

## 2. Inspect Model Architecture
We verify layer shapes and parameter counts.

In [ ]:
try:
    import torch
    model = create_issue_cnn(vocab_size=len(vocab), num_classes=4)
    print(model)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Total Trainable Parameters: {total_params:,}')
except ImportError:
    print('PyTorch not yet installed in local environment.')